# APIM ❤️ AI Agents

## Simplified ML Models to MCP lab

This focused lab keeps only the minimum setup needed to expose an Azure ML online endpoint as an MCP server through Azure API Management.

### What this lab deploys

- An Azure ML workspace and managed online endpoint for a pre-trained forecasting model
- An APIM REST API that forwards prediction requests to the Azure ML endpoint using managed identity
- An MCP server that exposes the `predict-forecast` operation as an MCP tool

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`.

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources are suffixed automatically by a unique string based on your subscription and resource group.
- Adjust the location parameters to your preferences.
- This simplified lab keeps only the Azure ML endpoint, the APIM REST API, and the MCP server.

In [1]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = "swedencentral"

apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

aml_endpoint_name_prefix = "forecast-endpoint"
aml_model_name = "forecast-model"
aml_deployment_name = "forecast-deployment"

subscription_id = utils.get_current_subscription()
utils.print_info(f"Using Subscription ID: {subscription_id}")
utils.print_ok('Notebook initialized')

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 06:36:58.518291 :0s]
👉🏽 Using Subscription ID: 784e6c68-d702-4a8b-a678-2e0f2f36deab (Visual Studio Enterprise Subscription)
👉🏽 Using Subscription ID: 784e6c68-d702-4a8b-a678-2e0f2f36deab
✅ Notebook initialized ⌚ 06:36:58.518291 


<a id='1'></a>
### 1️⃣ Verify the Azure CLI and install the ML extension

The following commands verify the current Azure login context and ensure that the `ml` Azure CLI extension is available.

In [2]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

ext_check = utils.run("az extension show --name ml", "", "", print_output=False)
if not ext_check.success:
    utils.run("az extension add --name ml -y", "Azure ML extension installed", "Failed to install Azure ML extension")
else:
    utils.print_ok("Azure ML extension already installed")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 06:37:10.583530 :0s]
👉🏽 Current user: baoqger@gmail.com
👉🏽 Tenant ID: 293882e2-ff46-4747-b46a-a84ca8e98fa0
👉🏽 Subscription ID: 784e6c68-d702-4a8b-a678-2e0f2f36deab
⚙️ Running: az extension show --name ml 
✅ Azure ML extension already installed ⌚ 06:37:11.171123 


<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to define the resources declaratively. The Bicep deployment creates the Azure ML workspace resources, the empty online endpoint, the APIM REST API, and the MCP server wrapper.

In [8]:
utils.create_resource_group(resource_group_name, resource_group_location)

bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": {"value": apim_sku},
        "apimSubscriptionsConfig": {"value": apim_subscriptions_config},
        "amlEndpointName": {"value": aml_endpoint_name_prefix}
    }
}

with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

deployment_output = utils.run(
    f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed"
)

if not deployment_output.success:
    raise RuntimeError(
        f"Deployment '{deployment_name}' failed. Fix the Bicep or Azure deployment error above, then rerun this cell."
    )

⚙️ Running: az group show --name lab-simplified-ml-models 
👉🏽 Using existing resource group 'lab-simplified-ml-models'
⚙️ Running: az deployment group create --name simplified-ml-models --resource-group lab-simplified-ml-models --template-file main.bicep --parameters params.json 
✅ Deployment 'simplified-ml-models' succeeded ⌚ 07:11:36.660700 :18s]


<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the Azure ML workspace details and the APIM MCP endpoint from the Bicep deployment.

In [9]:
output = utils.run(
    f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}",
    f"Failed to retrieve deployment: {deployment_name}"
)

if not output.success or not output.json_data:
    raise RuntimeError(f"Unable to retrieve deployment '{deployment_name}'.")

deployment_state = output.json_data.get('properties', {}).get('provisioningState')
if deployment_state != 'Succeeded':
    raise RuntimeError(
        f"Deployment '{deployment_name}' is in state '{deployment_state}'. Rerun the deployment cell after fixing the error above."
    )

apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
ml_prediction_api_endpoint = utils.get_deployment_output(output, 'mlPredictionApiEndpoint', 'ML Prediction API Endpoint')
aml_workspace_name = utils.get_deployment_output(output, 'amlWorkspaceName', 'Azure ML Workspace')
aml_endpoint_name_output = utils.get_deployment_output(output, 'amlEndpointName', 'Azure ML Endpoint')
mcp_endpoint = utils.get_deployment_output(output, 'mcpEndpoint', 'MCP Endpoint')
apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("'", '"'))
for subscription in apim_subscriptions:
    subscription_name = subscription['name']
    subscription_key = subscription['key']
    utils.print_info(f"Subscription Name: {subscription_name}")
    utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
utils.print_info('The ML Prediction API and MCP endpoint in this lab do not require subscription keys.')

⚙️ Running: az deployment group show --name simplified-ml-models -g lab-simplified-ml-models 
✅ Retrieved deployment: simplified-ml-models ⌚ 07:25:51.882140 :3s]
👉🏽 APIM Service Id: /subscriptions/784e6c68-d702-4a8b-a678-2e0f2f36deab/resourceGroups/lab-simplified-ml-models/providers/Microsoft.ApiManagement/service/apim-lqk2abkkuzazg
👉🏽 APIM API Gateway URL: https://apim-lqk2abkkuzazg.azure-api.net
👉🏽 ML Prediction API Endpoint: https://apim-lqk2abkkuzazg.azure-api.net/ml-prediction
👉🏽 Azure ML Workspace: aml-lqk2abkkuzazg
👉🏽 Azure ML Endpoint: forecast-endpointlqk2abkkuzazg
👉🏽 MCP Endpoint: https://apim-lqk2abkkuzazg.azure-api.net/ml-prediction-mcp/mcp
👉🏽 Subscription Name: subscription1
👉🏽 Subscription Key: ****c752
👉🏽 The ML Prediction API and MCP endpoint in this lab do not require subscription keys.


<a id='4'></a>
### 4️⃣ Register the ML model and create a deployment

The Bicep deployment creates an empty Azure ML online endpoint with AAD token authentication. This step registers the pre-trained MLflow model from `mlflow-model/`, creates the managed online deployment, and routes 100% of traffic to it.

In [11]:
import yaml, time

model_output = utils.run(
    f"az ml model create --name {aml_model_name} --path ./mlflow-model --type mlflow_model "
    f"--resource-group {resource_group_name} --workspace-name {aml_workspace_name} "
    f"--query version -o tsv",
    f"Model '{aml_model_name}' registered successfully",
    f"Failed to register model '{aml_model_name}'"
)
model_version = model_output.text.strip().split('\n')[-1].strip() if model_output.success else '1'
utils.print_info(f"Using model version: {model_version}")

sleep_time = 5
utils.print_info(f"Waiting for {sleep_time} seconds to ensure the model is fully registered before deployment...")
time.sleep(sleep_time)

⚙️ Running: az ml model create --name forecast-model --path ./mlflow-model --type mlflow_model --resource-group lab-simplified-ml-models --workspace-name aml-lqk2abkkuzazg --query version -o tsv 
✅ Model 'forecast-model' registered successfully ⌚ 09:14:26.754366 :25s]
👉🏽 Using model version: 1
👉🏽 Waiting for 5 seconds to ensure the model is fully registered before deployment...


In [ ]:
deployment_config = {
    "$schema": "https://azuremlschemas.azureedge.net/latest/managedOnlineDeployment.schema.json",
    "name": aml_deployment_name,
    "endpoint_name": aml_endpoint_name_output,
    "model": f"azureml:{aml_model_name}:{model_version}",
    "instance_type": "Standard_D2as_v4",
    "instance_count": 1,
}

with open('deployment.yml', 'w') as f:
    yaml.dump(deployment_config, f, default_flow_style=False)

utils.run(
    f"az ml online-deployment create --file deployment.yml "
    f"--resource-group {resource_group_name} --workspace-name {aml_workspace_name}",
    f"Deployment '{aml_deployment_name}' created successfully",
    f"Failed to create deployment '{aml_deployment_name}'"
)

utils.run(
    f"az ml online-endpoint update --name {aml_endpoint_name_output} --traffic \"{aml_deployment_name}=100\" "
    f"--resource-group {resource_group_name} --workspace-name {aml_workspace_name}",
    f"Traffic set to 100% for '{aml_deployment_name}'",
    f"Failed to update traffic"
)

⚙️ Running: az ml online-deployment create --file deployment.yml --resource-group lab-simplified-ml-models --workspace-name aml-lqk2abkkuzazg 
❌ Failed to create deployment 'forecast-deployment' ⌚ 09:21:21.371213 :17s] Check: endpoint forecast-endpointlqk2abkkuzazg exists
......................ERROR: (OutOfQuota) Not enough subscription CPU quota. The amount of CPU quota requested is 8 and your maximum amount of quota is [N/A]. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-outofquota
Code: OutOfQuota
Message: Not enough subscription CPU quota. The amount of CPU quota requested is 8 and your maximum amount of quota is [N/A]. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-outofquota

⚙️ Running: az ml online-endpoint update --name forecast-endpointlqk2abkkuzazg --traffic "forecast-deployment=100" --resource-group lab-simplified-ml-models --workspace-name aml-lqk2abkkuzazg 
❌ Failed to update traffic ⌚ 09:21:36.426314 :15s] ER

<a id='5'></a>
### 🧪 Test the MCP server connection and list tools

Connect to the MCP server and verify that the `predict-forecast` tool is available.

In [ ]:
import nest_asyncio
import asyncio
nest_asyncio.apply()

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


async def list_tools(server_url):
    async with streamablehttp_client(server_url) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            print(f"Available tools: {[tool.name for tool in tools.tools]}")
            for tool in tools.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(list_tools(mcp_endpoint))

### 🗑️ Clean up resources

When you're finished with the lab, run the [clean-up-resources notebook](clean-up-resources.ipynb) to remove the deployed resources.